In [ ]:
# =========================================================================================
# 📚 [튜토리얼] 군사 역사 기사 데이터 분석 실습: '시대를 읽는 키워드 찾기'
# ✍️ 데이터셋 이름: SinclairSchneider/military_times
# 💡 의미: 군대 및 군사 관련 역사 기사 모음입니다. (Military Times)
# 🎯 목표: 초보자를 위한 텍스트 분석 기초 실습입니다. 기사 제목과 본문에서 흥미로운 키워드를 추출하고,
#     실제 LLM 프롬프트 생성 원리를 맛보는 것을 목표로 합니다.
# -----------------------------------------------------------------------------------------
# 🌟 이 실습에서는 실제 기사 본문을 분석하여 '시대별 트렌드'를 파악하는 기획을 해봅시다!
# =========================================================================================

import random
from datasets import load_dataset, Dataset
from typing import List

# --- [설정 상수] ---
DATASET_ID = "SinclairSchneider/military_times"
SAMPLE_COUNT = 50  # 성능 테스트를 위해 상위 50개 샘플만 사용합니다!

# -----------------------------------------------------------------------------------------
# ⚙️ 1단계: 데이터 로딩 (스트리밍 vs. 일반 모드 처리)
# -----------------------------------------------------------------------------------------

# 초기 데이터셋 객체를 저장할 변수
dataset = None

print("🤖 튜터: 안녕! 오늘의 탐험할 데이터셋은 '군사 역사 기사'야. 데이터 로드부터 시작해 보자!")

try:
    # 🚩 1-1. 스트리밍 모드로 로딩 시도 (최신, 효율적인 방법!)
    # 데이터를 메모리에 한 번에 다 올리지 않고, 필요한 만큼만 가져와서 처리해요.
    print(f"✅ 시도: 스트리밍 모드({DATASET_ID}, split='train')로 로딩을 시작합니다...")
    dataset = load_dataset(DATASET_ID, split='train', streaming=True)
    print("✨ 성공! 스트리밍 모드로 데이터셋을 확보했습니다. 메모리 걱정 없이 대용량 데이터를 다룰 수 있어요!")

except Exception as e:
    # ⚠️ 스트리밍 모드 로딩에 실패했을 경우 (네트워크 문제 등)
    print(f"😔 아쉬워요! 스트리밍 로딩에 실패했습니다. ({e.__class__.__name__})")
    print("✨ 해결: 대신 소량만 로드하는 일반 모드로 대체합니다.")
    try:
        # 🚩 1-2. 일반 모드로 로딩 대체 (스트리밍=False)
        dataset = load_dataset(DATASET_ID, split='train', streaming=False)
        print("💾 성공! 소규모 데이터셋으로 대체 로딩을 완료했습니다. 이제 실습할 준비 끝!")
    except Exception as load_e:
        print(f"❌ 치명적인 오류 발생: 데이터셋 로드 자체가 불가능합니다. (오류: {load_e})")
        exit()

# -----------------------------------------------------------------------------------------
# 🔬 2단계: 데이터 샘플링 및 준비
# -----------------------------------------------------------------------------------------

print("\n=============================================================")
print("🔬 2단계: 데이터 샘플 준비 (상위 {}개만 사용합니다!)".format(SAMPLE_COUNT))

# 🚀 필수 요구조건에 맞춰, streaming 모드와 일반 모드에 관계없이 안정적으로 상위 K개를 가져오는 패턴을 사용합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)인 경우
    print("➡️ 스트리밍 패턴 감지: dataset.take()를 사용하여 샘플을 추출합니다.")
    # list(dataset.take(K))를 통해 반드시 리스트 형태의 Iterator로 만듭니다.
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)인 경우
    print("➡️ 일반 패턴 감지: list(dataset.select(...)) 패턴을 사용합니다.")
    # 만약 일반 Dataset이라면, 처음 K개를 잘라내어 사용합니다.
    sampled_dataset = dataset.select(range(min(SAMPLE_COUNT, len(dataset))))
    # 반복 처리를 위해 리스트로 변환합니다.
    sampled_dataset_iterator = iter(sampled_dataset)

# 효율적인 처리를 위해 처음부터 리스트로 변환합니다.
sample_data_list: List[dict] = list(sampled_dataset_iterator)

print("✨ 준비 완료! 상위 {}개의 데이터를 메모리에 로드하여 분석을 시작합니다.".format(len(sample_data_list)))

# -----------------------------------------------------------------------------------------
# 💻 3단계: 실습 1 - 데이터 구조 분석 및 통계 (친절한 데이터 탐색)
# -----------------------------------------------------------------------------------------

print("\n\n=============================================================")
print("📊 실습 1: 데이터 구조 분석 및 통계 (기사 제목 길이 분석)");
print("=============================================================")

# 💡 목표: 가장 짧거나 긴 기사의 특징을 찾아보고, 데이터가 어떤 구조로 되어 있는지 눈으로 익혀보세요!

print("\n[🔍 구조 확인]: 첫 번째 샘플의 데이터 구조를 살펴봅시다.")
if sample_data_list:
    first_sample = sample_data_list[0]
    print(f"  -> Title (기사 제목): {first_sample.get('title', 'N/A')[:50]}...")
    print(f"  -> Content (본문 내용): {first_sample.get('content', 'N/A')[:50]}...")

# 통계 분석: 모든 기사의 제목 길이 평균과 표준편차를 계산해봅시다.
title_lengths = []
for sample in sample_data_list:
    title = sample.get('title', '')
    title_lengths.append(len(title))

if title_lengths:
    avg_length = sum(title_lengths) / len(title_lengths)
    min_length = min(title_lengths)
    max_length = max(title_lengths)

    print("\n⭐ 기사 제목 길이 통계 분석:")
    print(f"  ✅ 총 분석된 기사 수: {len(title_lengths)}개")
    print(f"  📏 평균 제목 길이: {avg_length:.2f} 자")
    print(f"  🔽 가장 짧은 제목 길이: {min_length} 자")
    print(f"  🔼 가장 긴 제목 길이: {max_length} 자")
    print("  (힌트: 데이터가 일정하지 않죠? 이 때문에 키워드 분석이 중요해요!)")
else:
    print("⚠️ 경고: 분석할 데이터가 없습니다.")


# -----------------------------------------------------------------------------------------
# ✍️ 4단계: 실습 2 - 키워드 추출 시뮬레이션 (NLP 초심자 과정)
# -----------------------------------------------------------------------------------------

print("\n\n=============================================================")
print("✍️ 실습 2: 키워드 추출 시뮬레이션 (가장 많이 쓰인 단어 찾기)");
print("=============================================================")

from collections import Counter
import re

# 💡 목표: '제목'과 '본문'을 합쳐서, 일반적인 단어(stop words)를 제외한 유의미한 단어들을 찾아봅시다.
# (간단한 NLP 처리를 통해 '핵심 키워드'를 추출하는 원리를 이해하는 것이 목표입니다!)

def simple_keyword_extractor(text: str) -> List[str]:
    """텍스트를 전처리하고 키워드 후보를 리스트로 반환합니다."""
    # 1. 소문자 변환 및 특수문자 제거
    text = text.lower()
    cleaned_text = re.sub(r'[^\w\s]', '', text)
    # 2. 구두점이나 숫자가 붙은 것 제거
    words = cleaned_text.split()
    
    # 3. 간단한 불용어(Stop Words) 필터링 (실제로는 더 복잡합니다!)
    stop_words = {"a", "the", "is", "and", "of", "to", "in", "it", "by", "for", "with"}
    keywords = [word for word in words if word not in stop_words and len(word) > 2]
    return keywords

# 모든 기사에서 키워드를 모읍니다.
all_keywords = []
for sample in sample_data_list:
    title = sample.get('title', '')
    content = sample.get('content', '')
    
    # 제목과 본문 모두에서 키워드를 뽑아모읍니다.
    title_keywords = simple_keyword_extractor(title)
    content_keywords = simple_keyword_extractor(content)
    
    all_keywords.extend(title_keywords)
    all_keywords.extend(content_keywords)

# 가장 많이 등장한 상위 10개 키워드 계산
keyword_counts = Counter(all_keywords)
top_keywords = keyword_counts.most_common(10)

print("\n⭐ 전체 데이터에서 가장 많이 반복된 상위 10개 키워드:")
for keyword, count in top_keywords:
    print(f"  -> '{keyword}': 약 {count}회 등장 (이 단어가 기사의 핵심 주제일 확률이 높아요!)")


# -----------------------------------------------------------------------------------------
# 🛠️ 5단계: 실습 3 - LLM 프롬프트 생성 실습 (미래의 AI와 대화하는 법)
# -----------------------------------------------------------------------------------------

print("\n\n=============================================================")
print("🤖 실습 3: LLM 프롬프트 작성 실습 (프롬프트 엔지니어링 맛보기)");
print("=============================================================")

# 💡 목표: 분석된 데이터를 가지고 'AI에게 던질 최고의 질문'을 만드는 방법을 배웁니다.
# 데이터가 분석된 만큼, 단순히 내용을 요약해달라고 하기보다 '역할을 부여'해주는 것이 중요해요!

if sample_data_list:
    # 임의의 샘플 1개를 선택합니다.
    target_sample = random.choice(sample_data_list)
    target_title = target_sample.get('title', '제목 없음')
    target_content = target_sample.get('content', '내용 없음')

    print("\n[🎯 분석 대상 기사 (샘플 선택)]")
    print(f"  * 제목: {target_title[:80]}...")
    print(f"  * 본문: {target_content[:80]}...")

    # 프롬프트 템플릿 생성 (여기가 바로 AI의 성능을 좌우합니다!)
    prompt_template = f"""
    [역할 부여]: 당신은 20세기 군사 역사 전문 저널리스트이자, 고고학자입니다.
    [요청]: 다음 기사를 분석하여, 이 사건의 가장 중요한 '배경 지식(Context)' 3가지와, 이 사건이 후대에 미친 '장기적인 영향(Impact)' 2가지를 각각 bullet point로 요약해 주세요.
    [분석 대상 기사 내용]:
    제목: {target_title}
    본문: {target_content}
    """
    
    print("\n✅ 생성된 고품질 LLM 프롬프트 (AI에 질문할 내용):")
    print("-" * 60)
    print(prompt_template.strip())
    print("-" * 60)
    print("\n✨ 튜터 코멘트: 이렇게 역할을 구체적으로 부여하고 (저널리스트), 원하는 결과물의 형식(bullet point, 3가지, 2가지)까지 지정해 주면, AI는 훨씬 더 쓸모 있는 답변을 해줄 거예요! 이것이 '프롬프트 엔지니어링'의 핵심입니다!")
else:
    print("⚠️ 실습 3을 진행할 데이터가 부족합니다.")

print("\n\n=============================================================")
print("🎉 축하합니다! 모든 실습을 완료했습니다! 데이터의 숨겨진 이야기를 찾아내는 능력, 최고예요! 😊")
print("=============================================================")